# 2. Counting Abstract & Concrete Words

Score passages of text by sliding a window and counting how many recognized words
fall into abstract, concrete, or neither categories.

In [ ]:
import pandas as pd
from abstraction.counting import count_absconc, count_absconc_psg
from abstraction.scoring import score_psg

## Score a single passage

In [ ]:
txt = """It was a dark and stormy night; the rain fell in torrents — except 
at occasional intervals, when it was checked by a violent gust of wind which 
swept up the streets (for it is in London that our scene lies), rattling along 
the housetops, and fiercely agitating the scanty flame of the lamps that 
struggled against the darkness."""

score = score_psg(txt)
print(f'Mean concreteness score: {score:.3f}')
print('(negative = more abstract, positive = more concrete)')

## Window-based counting

Slide a window of 100 recognized words across a text, counting abstract/concrete in each window.

In [ ]:
# A longer passage for windowed counting
austen = """Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings of existence;
and had lived nearly twenty-one years in the world with very little to distress
or vex her. She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage, been mistress
of his house from a very early period. Her mother had died too long ago for her
to have more than an indistinct remembrance of her caresses; and her place had
been supplied by an excellent woman as governess, who had fallen little short of
a mother in affection. Sixteen years had Miss Taylor been in Mr. Woodhouse's
family, less as a governess than a friend, very fond of both daughters, but
particularly of Emma. Between them it was more the intimacy of sisters. Even
before Miss Taylor had ceased to hold the nominal office of governess, the
mildness of her temper had hardly allowed her to impose any restraint; and the
shadow of authority being now long passed away, they had been living together as
friend and friend very mutually attached, and Emma doing just what she liked;
highly esteeming Miss Taylor's judgment, but directed chiefly by her own."""

results = count_absconc(austen)
df = pd.DataFrame(results)
if len(df):
    df['abs-conc'] = df['num_abs'] - df['num_conc']
    print(f'{len(df)} windows')
    df[['slice', 'num_abs', 'num_conc', 'num_neither', 'num_total', 'abs-conc']]
else:
    print('Text too short for a full window (need 100 recognized words)')

## Counting with passage markup

`count_absconc_psg` includes HTML-marked passages showing which words were classified.

In [ ]:
df_psg = count_absconc_psg(austen)
if len(df_psg):
    print(f'{len(df_psg)} windows')
    df_psg[['slice', 'num_abs', 'num_conc', 'abs-conc']]
else:
    print('Text too short for windowed counting')

In [ ]:
# Display a marked-up passage
if len(df_psg):
    from IPython.display import HTML
    row = df_psg.iloc[0]
    display(HTML(f'<p>{row["passage"]}</p>'))
    print(f'\nAbstract: {row["num_abs"]}, Concrete: {row["num_conc"]}, Abs-Conc: {row["abs-conc"]}')

## Compare two passages

In [ ]:
abstract_txt = """The principle of justice demands that we consider the moral
implications of our decisions with care and deliberation. Freedom, equality,
and dignity are not merely abstract ideals but constitute the foundation of
civilized society. The obligation to respect these values transcends the
particular circumstances of any individual case."""

concrete_txt = """She picked up the heavy iron skillet from the wooden counter
and cracked two brown eggs into the sizzling butter. The kitchen smelled of
bacon and coffee. Outside the window, a red cardinal perched on the oak branch,
its feathers bright against the grey morning sky."""

print(f'Abstract passage score: {score_psg(abstract_txt):.3f}')
print(f'Concrete passage score: {score_psg(concrete_txt):.3f}')

## Corpus-level counting

To count across an entire corpus (e.g. CanonFiction), use `count_absconc_corpus`.
This runs in parallel and writes results to a CSV.

In [ ]:
# Uncomment to run (slow — processes all texts in the corpus):
# from abstraction.counting import count_absconc_corpus
# count_absconc_corpus('CanonFiction', num_proc=4)